# LoRA finetune
Finetunes encoder with LoRA adapters, reuses existing logic from `embed_dataset.py` embedding pipeline (chunking + averaging) + tweak to pass finetuned encoder instead of pretrained to produce embeddings from the finetuned encoder

In [ ]:
# Install dependencies
%pip install -q transformers datasets accelerate peft evaluate scikit-learn sentence-transformers importlib-metadata

import peft
import transformers
import torch

print("PEFT version:", peft.__version__)
print("Transformers version:", transformers.__version__)
print("Torch version:", torch.__version__)

In [ ]:
# Config - edit these before running (MODEL_NAME and DATA_CSV are required)
MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2'
DATA_CSV = 'data/your_dataset.csv'  # set path to your CSV file
# Output folders (each run will create subfolders under these)
ADAPTER_DIR = '/kaggle/working/runs/lora/lora_r8_a16_bs8_lr1e-4_ep3'
OUT_DIR = '/kaggle/working/runs/lora/embeddings_from_lora'
# Training defaults - tweak per-run via grid_options in the experiment runner
TRAIN_FP16 = True
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 32
NUM_EPOCHS = 3
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 1e-4
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
# Embedding batch size used by load_and_embed when saving embeddings
BATCH_SIZE = 32

### What's a LoRA adapter?
LoRA (Low-Rank Adaptation) injects a small set of additional, trainable parameters into a large model (typically into attention or projection matrices) -- instead of updating all model weights, LoRA only trains these small adapter matrices (rank `r`) which are much cheaper to store and train. After training, can save just the adapter weights and later reapply them to the original base model.

**Kaggle run tips**
- Set Notebook runtime to GPU (Kaggle: Settings → Accelerator → GPU).
- Set `TRAIN_FP16 = True` in the config when running on GPU to reduce memory and speed up training.
- Recommended starting values: `TRAIN_BATCH_SIZE=16` (if GPU memory allows), `NUM_EPOCHS=3`.
- If you prefer script mode, install dependencies then run with `accelerate`:

```bash
pip install -q transformers datasets accelerate peft
# configure accelerate once interactively or use defaults
accelerate launch your_train_script.py
```

This notebook is already configured to save adapters to `ADAPTER_DIR` and then load them for embedding later in the notebook.

In [ ]:
# LoRA training (supervised classification) -- creates and saves adapters to ADAPTER_DIR
# This cell reads the CSV, builds a HF Dataset, trains with PEFT+Trainer, and saves the adapter
from datasets import Dataset
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
import os
import inspect

# Read CSV and build `text`, `label`, and `num_turns` columns to match embed_dataset expansion (k=1..n)
df = pd.read_csv(DATA_CSV, dtype=str).fillna("")
output_cols = sorted([c for c in df.columns if c.startswith('llm_output_')], key=lambda s: int(s.split('_')[-1]) if s.split('_')[-1].isdigit() else s)
rows = []
for _, row in df.iterrows():
    trait = row.get('trait_selected', row.get('trait', '0'))
    try:
        label = int(trait) if trait != '' else 0
    except Exception:
        label = 0
    outputs = []
    for col in output_cols:
        txt = row.get(col, '')
        if txt is None:
            txt = ''
        txt = str(txt).strip()
        if txt != '':
            outputs.append(txt)
    if len(outputs) == 0:
        continue
    # Expand each conversation into multiple examples (k=1..len(outputs)) to match embed_dataset
    for k in range(1, len(outputs) + 1):
        text = '\n\n'.join(outputs[:k])
        rows.append({'text': text, 'label': label, 'num_turns': int(k)})

meta_df = pd.DataFrame(rows)

labels = sorted(meta_df['label'].unique())
num_labels = len(labels)
label2id = {int(l): i for i, l in enumerate(labels)}
meta_df['label_id'] = meta_df['label'].astype(int).map(label2id)

# Try to stratify by (label_id, num_turns) to preserve distribution across convo lengths; fall back to label-only if not possible
stratify_col = None
try:
    stratify_col = meta_df['label_id'].astype(str) + '_' + meta_df['num_turns'].astype(str)
    train_df, eval_df = train_test_split(meta_df, test_size=0.15, stratify=stratify_col, random_state=42)
except Exception:
    train_df, eval_df = train_test_split(meta_df, test_size=0.15, stratify=meta_df['label_id'], random_state=42)

train_ds = Dataset.from_pandas(train_df[['text','label_id']].rename(columns={'label_id':'label'}))
eval_ds = Dataset.from_pandas(eval_df[['text','label_id']].rename(columns={'label_id':'label'}))

# Tokenizer + map
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
max_length = 256
def tokenize_fn(batch):
    return hf_tokenizer(batch['text'], padding='max_length', truncation=True, max_length=max_length)
train_ds = train_ds.map(tokenize_fn, batched=True)
eval_ds = eval_ds.map(tokenize_fn, batched=True)
train_ds = train_ds.remove_columns(['text'])
eval_ds = eval_ds.remove_columns(['text'])
train_ds.set_format(type='torch')
eval_ds.set_format(type='torch')

# Load model for sequence classification, then apply LoRA (PEFT)
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

# Diagnostic: list torch.nn.Linear module names to assist choosing PEFT target_modules
import torch as _torch
linear_names = [name for name, m in base_model.named_modules() if isinstance(m, _torch.nn.Linear)]
print('Detected', len(linear_names), 'Linear modules. Sample names:')
for n in linear_names[:80]:
    print('  ', n)

# Auto-select PEFT target module name fragments that match MPNet naming.
# Prefer explicit attention projection fragments when present, else fall back to
# a small set of common linear fragments so PEFT can find modules by substring.
preferred_frags = ['attention.attn.q', 'attention.attn.k', 'attention.attn.v', 'attention.attn.o', 'attn.q', 'attn.k', 'attn.v', 'attn.o']
found_frags = [f for f in preferred_frags if any(f in n for n in linear_names)]

if found_frags:
    # Use the final fragment (e.g., 'q','k','v','o' or 'attn.q') where available
    # PEFT matches by substring, so shorter fragments often work well.
    target_modules = list({f.split('.')[-1] for f in found_frags})
else:
    # Fallback fragments (generic names that appear in many classifier/transformer Linear modules)
    target_modules = ['attn', 'attention', 'dense', 'output', 'projection', 'query', 'key', 'value']

# Include classifier head layers if present (optional but useful for seq-class tasks)
if any('classifier.dense' in n for n in linear_names):
    target_modules += ['classifier.dense', 'classifier.out_proj']

print('Using PEFT target_modules fragments:', target_modules)

# Create LoraConfig with the selected fragments
lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=target_modules, lora_dropout=0.05, bias='none', task_type=TaskType.SEQ_CLS)
model = get_peft_model(base_model, lora_config)

# Detect device (CUDA -> DirectML -> CPU)
import importlib
if _torch.cuda.is_available():
    device = 'cuda'
else:
    try:
        td = importlib.import_module('torch_directml')
        device = td.device()
    except Exception:
        device = 'cpu'

# Move model to device
model.to(device)

# Training arguments - construct kwargs dynamically for compatibility across transformers versions
training_kwargs = dict(
    output_dir=os.path.join(ADAPTER_DIR),
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    save_strategy='epoch',
    learning_rate=1e-4,
    fp16=TRAIN_FP16,
    logging_steps=50,
)
# Add evaluation strategy/eval flag only if supported by installed transformers
try:
    sig = inspect.signature(TrainingArguments)
    params = set(sig.parameters.keys())
    if 'evaluation_strategy' in params:
        training_kwargs['evaluation_strategy'] = 'epoch'
    elif 'evaluate_during_training' in params:
        training_kwargs['evaluate_during_training'] = True
except Exception:
    training_kwargs['evaluation_strategy'] = 'epoch'

training_args = TrainingArguments(**training_kwargs)

from sklearn.metrics import accuracy_score, f1_score
def compute_metrics(p):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    preds = np.argmax(preds, axis=1)
    labels = p.label_ids
    return {'accuracy': accuracy_score(labels, preds), 'f1_macro': f1_score(labels, preds, average='macro')}

# Build Trainer kwargs dynamically so we only pass supported params (some transformers versions reject 'tokenizer')
trainer_kwargs = dict(model=model, args=training_args, train_dataset=train_ds, eval_dataset=eval_ds, compute_metrics=compute_metrics)
try:
    t_sig = inspect.signature(Trainer)
    t_params = set(t_sig.parameters.keys())
    if 'tokenizer' in t_params:
        trainer_kwargs['tokenizer'] = hf_tokenizer
except Exception:
    # If introspection fails, cautiously avoid passing tokenizer
    pass

trainer = Trainer(**trainer_kwargs)

# Run training (this trains only LoRA params because other params are frozen by PEFT by default)
trainer.train()

# Save adapters / peft weights to ADAPTER_DIR so later cells can load them for embedding
model.save_pretrained(ADAPTER_DIR)
print('LoRA adapters saved to', ADAPTER_DIR)


## Pass finetuned model
Load base HF model and wrap it with the LoRA adapters (PEFT), then pass to `embed_dataset.py` embedding logic

In [ ]:
# Load HF + PEFT model and set hf_model/hf_tokenizer used by embed logic
import importlib
import torch
from transformers import AutoTokenizer, AutoModel
from peft import PeftModel
from sentence_transformers import SentenceTransformer

# Detect best available device: CUDA -> DirectML -> CPU
if torch.cuda.is_available():
    device = "cuda"
else:
    try:
        td = importlib.import_module("torch_directml")
        device = td.device()          # torch_directml.device() works with torch_directml
    except Exception:
        device = "cpu"

# SentenceTransformer instance used for tokenization and max-length helpers (same as embed_dataset does)
st_model = SentenceTransformer(MODEL_NAME, device=device)

# Load HF tokenizer and base AutoModel, then wrap with PEFT adapters saved in ADAPTER_DIR
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base = AutoModel.from_pretrained(MODEL_NAME)
# PeftModel.from_pretrained wraps the base model and applies the adapter weights
peft_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
peft_model.to(device)
peft_model.eval()

# Embed logic expects hf_model and hf_tokenizer as params -- just set these to the finetuned versions to embed using the finetuned encoder
print('Loaded PEFT model and tokenizer; ready to embed using finetuned encoder')

In [ ]:
# embed_dataset.py logic
import numpy as np, os
from transformers import AutoTokenizer, AutoModel
import torch

# HF tokenizer/model lazily used only to avoid inference-mode tensor issues
# hf_tokenizer and hf_model set in cell above (PEFT-wrapped)
# If not set, fall back to loading the base model
# Use `# noqa` to silence style/lint checks when a line is intentionally atypical
try:
    hf_tokenizer  # noqa
except NameError:
    print("HF tokenizer not found, loading base model tokenizer...")
    hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

try:
    hf_model  # noqa
except NameError:
    print("HF model not found, loading base model...")
    hf_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
    hf_model.eval()

# Reuse st_model (SentenceTransformer) for tokenization and max seq length helpers
model = st_model

def chunk_and_aggregate(text) :
    # Use HuggingFace tokenizer to split by tokens -- chunks should overlap a little bit, but not too much (maybe 30 token overlap?)
    tokens = model.tokenize(text) # returns dict, NOT list of tokens
    token_list = tokens['input_ids'][0].tolist() # get list of token ids from batch of 1
    
    # Max length of tokens for this model
    max_length = model.get_max_seq_length()
    
    chunks = [token_list[i:i+max_length] for i in range(0, len(token_list), max_length - 30)]  # 30 token overlap

    # Decode token-id chunks back to text before calling `model.encode` to avoid
    # creating inference-mode tensors from raw id lists (which can trigger
    # "Cannot set version_counter for inference tensor" errors with some models).
    tokenizer = None
    try:
        tokenizer = model._first_module().tokenizer
    except Exception:
        tokenizer = None

    if tokenizer is not None:
        chunk_texts = [tokenizer.decode(c, skip_special_tokens=True) for c in chunks]
    else:
        chunk_texts = [" ".join(map(str, c)) for c in chunks]

    # Get embeddings for each chunk and average them.
    # Use HF AutoModel + tokenizer under torch.no_grad() to avoid creating
    # inference-mode tensors that some MPNet internals mutate.
    global hf_tokenizer, hf_model
    if hf_tokenizer is None or hf_model is None:
        hf_tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-mpnet-base-v2')
        hf_model = AutoModel.from_pretrained('sentence-transformers/all-mpnet-base-v2')
        hf_model.to(device)

    tokens = hf_tokenizer(chunk_texts, padding=True, truncation=True, return_tensors='pt')
    tokens = {k: v.to(device) for k, v in tokens.items()}
    with torch.no_grad():
        out = hf_model(**tokens, return_dict=True)
    last_hidden = out.last_hidden_state  # (batch, seq_len, dim)
    mask = tokens['attention_mask'].unsqueeze(-1).to(last_hidden.dtype)
    pooled = (last_hidden * mask).sum(1) / mask.sum(1)
    chunk_embeddings = pooled.cpu().numpy()
    aggregated_embedding = chunk_embeddings.mean(axis=0)
    
    return aggregated_embedding


def embed_batch(batch):
    embeddings = []
    labels = []
    metadata = [] # trait and num_turns -- for doing stratified split later in the observer logic

    # DataLoader collates batch into a tuple (texts_list, labels_list, ks_list) instead of list of tuples
    texts, labs, ks = batch
    for response, label, k in zip(texts, labs, ks):
        label_val = int(label.item()) if isinstance(label, torch.Tensor) else int(label)
        embedding = chunk_and_aggregate(response)
        embeddings.append(embedding)
        labels.append(label_val)
        metadata.append({'trait': label_val, 'num_turns': int(k)})
    
    # Stack into single ndarray first before converting to tensor, since list of np arrays --> tensor is super slow apparantly
    batch_embeddings = np.vstack(embeddings)
    batch_labels = np.array(labels, dtype=int)
    return torch.from_numpy(batch_embeddings), torch.from_numpy(batch_labels), metadata


# Embed all responses in the dataset and save to .npy files
import numpy as np
import os
def embed_dataset(dataloader, out_dir='embeddings') :
    os.makedirs(out_dir, exist_ok=True)
    
    all_embeddings = []
    all_labels = []
    metadata = []
    
    for batch in dataloader:
        batch_embeddings, batch_labels, batch_metadata = embed_batch(batch)
        all_embeddings.append(batch_embeddings)
        all_labels.append(batch_labels)
        metadata.append(batch_metadata)
        
    all_embeddings = torch.cat(all_embeddings).numpy()
    all_labels = torch.cat(all_labels).numpy()
    metadata = [item for sublist in metadata for item in sublist]
    
    np.save(os.path.join(out_dir, 'embeddings.npy'), all_embeddings)
    np.save(os.path.join(out_dir, 'labels.npy'), all_labels)
    
    # Save metadata as JSON
    import json
    
    # Metadata is a list of tensors, convert to list of dicts first
    metadata = [{'trait': int(m['trait']), 'num_turns': int(m['num_turns'])} for m in metadata]
    
    with open(os.path.join(out_dir, 'metadata.json'), 'w') as f:
        json.dump(metadata, f)
    
    
# Load responses from csv, concatenate responses for each convo length, and embed
import pandas as pd
def load_and_embed(csv_path, out_dir='embeddings', batch_size=32):
   
    df = pd.read_csv(csv_path, dtype=str).fillna("")

    # Get llm output columns (e.g. llm_output_0 .. llm_output_4)
    output_cols = sorted([c for c in df.columns if c.startswith('llm_output_')],
                         key=lambda s: int(s.split('_')[-1]) if s.split('_')[-1].isdigit() else s)

    items = []  # list of (concat_text, label)

    for _, row in df.iterrows():
        # Try trait_selected, fallback to trait or '0'
        trait = row.get('trait_selected', row.get('trait', '0'))
        try:
            label = int(trait) if trait != '' else 0
        except Exception:
            label = 0

        outputs = []
        for col in output_cols:
            txt = row.get(col, "")
            if txt is None:
                txt = ""
            txt = str(txt).strip()
            if txt != "":
                outputs.append(txt)

        if len(outputs) == 0:
            continue

        # create concatenated texts for each convo length, include num_turns in each item for stratified splitting later
        for k in range(1, len(outputs) + 1):
            concat = "\n\n".join(outputs[:k])
            items.append((concat, label, k))  # (text, label, num_turns)

    # Initialize dataloader and embed dataset
    dataloader = torch.utils.data.DataLoader(items, batch_size=batch_size, shuffle=False)
    embed_dataset(dataloader, out_dir)

In [ ]:
# Experiment runner: hyperparameter grid, train LoRA adapters, then embed with each adapter
import os, itertools, inspect, json, importlib
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from sentence_transformers import SentenceTransformer

# Detect device locally (CUDA -> DirectML -> CPU) so runner works standalone and under accelerate
if torch.cuda.is_available():
    device = 'cuda'
else:
    try:
        td = importlib.import_module('torch_directml')
        device = td.device()
    except Exception:
        device = 'cpu'

# Basic checks for required globals
if 'MODEL_NAME' not in globals():
    raise RuntimeError('Please set MODEL_NAME in the notebook before running the experiment runner')
if 'DATA_CSV' not in globals():
    raise RuntimeError('Please set DATA_CSV (path to your CSV) in the notebook before running the experiment runner')

# Hyperparameter grid: adjust lists below to try different values
grid_options = dict(
    LORA_R=[8],
    LORA_ALPHA=[16],
    TRAIN_BATCH_SIZE=[8],
    LEARNING_RATE=[1e-4],
    NUM_EPOCHS=[3],
)

def generate_configs(grid):
    keys = list(grid.keys())
    pools = [grid[k] for k in keys]
    for comb in itertools.product(*pools):
        yield dict(zip(keys, comb))

def _build_datasets(csv_path):
    df = pd.read_csv(csv_path, dtype=str).fillna("")
    output_cols = sorted([c for c in df.columns if c.startswith('llm_output_')], key=lambda s: int(s.split('_')[-1]) if s.split('_')[-1].isdigit() else s)
    rows = []
    for _, row in df.iterrows():
        trait = row.get('trait_selected', row.get('trait', '0'))
        try:
            label = int(trait) if trait != '' else 0
        except Exception:
            label = 0
        outputs = []
        for col in output_cols:
            txt = row.get(col, '')
            if txt is None:
                txt = ''
            txt = str(txt).strip()
            if txt != '':
                outputs.append(txt)
        if len(outputs) == 0:
            continue
        for k in range(1, len(outputs) + 1):
            text = '\n\n'.join(outputs[:k])
            rows.append({'text': text, 'label': label, 'num_turns': int(k)})
    meta_df = pd.DataFrame(rows)
    labels = sorted(meta_df['label'].unique())
    label2id = {int(l): i for i, l in enumerate(labels)}
    meta_df['label_id'] = meta_df['label'].astype(int).map(label2id)
    from sklearn.model_selection import train_test_split
    try:
        stratify_col = meta_df['label_id'].astype(str) + '_' + meta_df['num_turns'].astype(str)
        train_df, eval_df = train_test_split(meta_df, test_size=0.15, stratify=stratify_col, random_state=42)
    except Exception:
        train_df, eval_df = train_test_split(meta_df, test_size=0.15, stratify=meta_df['label_id'], random_state=42)
    from datasets import Dataset
    train_ds = Dataset.from_pandas(train_df[['text','label_id']].rename(columns={'label_id':'label'}))
    eval_ds = Dataset.from_pandas(eval_df[['text','label_id']].rename(columns={'label_id':'label'}))
    return train_ds, eval_ds, len(labels)

def _auto_select_target_modules(model):
    import torch as _torch
    linear_names = [name for name, m in model.named_modules() if isinstance(m, _torch.nn.Linear)]
    preferred_frags = ['attention.attn.q', 'attention.attn.k', 'attention.attn.v', 'attention.attn.o', 'attn.q', 'attn.k', 'attn.v', 'attn.o']
    found_frags = [f for f in preferred_frags if any(f in n for n in linear_names)]
    if found_frags:
        target_modules = list({f.split('.')[-1] for f in found_frags})
    else:
        target_modules = ['attn', 'attention', 'dense', 'output', 'projection', 'query', 'key', 'value']
    if any('classifier.dense' in n for n in linear_names):
        target_modules += ['classifier.dense', 'classifier.out_proj']
    return target_modules

def _make_training_args(output_dir, per_device_train_batch_size, per_device_eval_batch_size, num_train_epochs, lr, fp16):
    kwargs = dict(
        output_dir=output_dir,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_eval_batch_size,
        num_train_epochs=num_train_epochs,
        save_strategy='epoch',
        learning_rate=lr,
        fp16=fp16,
        logging_steps=50,
    )
    try:
        sig = inspect.signature(TrainingArguments)
        params = set(sig.parameters.keys())
        if 'evaluation_strategy' in params:
            kwargs['evaluation_strategy'] = 'epoch'
        elif 'evaluate_during_training' in params:
            kwargs['evaluate_during_training'] = True
    except Exception:
        kwargs['evaluation_strategy'] = 'epoch'
    return TrainingArguments(**kwargs)

def run_experiments(grid_options, overwrite=False, embed_batch_size=None):
    configs = list(generate_configs(grid_options))
    print(f'Will run {len(configs)} configurations')
    for cfg in configs:
        print('---')
        print('Config:', cfg)
        r = cfg['LORA_R']
        alpha = cfg['LORA_ALPHA']
        bs = cfg['TRAIN_BATCH_SIZE']
        lr = cfg['LEARNING_RATE']
        epochs = cfg['NUM_EPOCHS']
        run_name = f'r{r}_a{alpha}_bs{bs}_lr{lr}_ep{epochs}'
        adapter_dir = os.path.join(ADAPTER_DIR, run_name)
        out_dir = os.path.join(OUT_DIR, run_name)
        os.makedirs(adapter_dir, exist_ok=True)
        os.makedirs(out_dir, exist_ok=True)

        if not overwrite and os.path.exists(os.path.join(adapter_dir, 'adapter_config.json')):
            print('Adapter already exists, skipping training:', adapter_dir)
        else:
            train_ds, eval_ds, num_labels = _build_datasets(DATA_CSV)
            hf_tokenizer_local = AutoTokenizer.from_pretrained(MODEL_NAME)
            max_length = 256
            def tokenize_fn(batch):
                return hf_tokenizer_local(batch['text'], padding='max_length', truncation=True, max_length=max_length)
            train_ds = train_ds.map(tokenize_fn, batched=True)
            eval_ds = eval_ds.map(tokenize_fn, batched=True)
            train_ds = train_ds.remove_columns(['text'])
            eval_ds = eval_ds.remove_columns(['text'])
            train_ds.set_format(type='torch')
            eval_ds.set_format(type='torch')

            base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
            target_modules = _auto_select_target_modules(base)
            print('Using target_modules fragments:', target_modules)
            lora_conf = LoraConfig(r=r, lora_alpha=alpha, target_modules=target_modules, lora_dropout=globals().get('LORA_DROPOUT', 0.05), bias='none', task_type=TaskType.SEQ_CLS)
            model = get_peft_model(base, lora_conf)

            training_args = _make_training_args(output_dir=adapter_dir, per_device_train_batch_size=bs, per_device_eval_batch_size=bs*4, num_train_epochs=epochs, lr=lr, fp16=TRAIN_FP16)

            from sklearn.metrics import accuracy_score, f1_score
            def compute_metrics(p):
                preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
                preds = np.argmax(preds, axis=1)
                labels = p.label_ids
                return {'accuracy': accuracy_score(labels, preds), 'f1_macro': f1_score(labels, preds, average='macro')}

            trainer_kwargs = dict(model=model, args=training_args, train_dataset=train_ds, eval_dataset=eval_ds, compute_metrics=compute_metrics)
            try:
                t_sig = inspect.signature(Trainer)
                t_params = set(t_sig.parameters.keys())
                if 'tokenizer' in t_params:
                    trainer_kwargs['tokenizer'] = hf_tokenizer_local
            except Exception:
                pass

            trainer = Trainer(**trainer_kwargs)
            trainer.train()
            model.save_pretrained(adapter_dir)
            print('Saved adapter to', adapter_dir)

        # Load adapter for embedding and run embedding pipeline into per-run folder
        base_for_embed = AutoModel.from_pretrained(MODEL_NAME)
        peft_model = PeftModel.from_pretrained(base_for_embed, adapter_dir)
        peft_model.to(device)
        peft_model.eval()
        # Set globals used by embed logic (hf_model, hf_tokenizer, st_model)
        globals()['hf_model'] = peft_model
        globals()['hf_tokenizer'] = AutoTokenizer.from_pretrained(MODEL_NAME)
        globals()['st_model'] = SentenceTransformer(MODEL_NAME, device=device)
        # Embed dataset using this adapter and save into run-specific out_dir
        batch_for_embed = embed_batch_size or globals().get('BATCH_SIZE', 32)
        load_and_embed(DATA_CSV, out_dir=out_dir, batch_size=batch_for_embed)
        print('Saved embeddings to', out_dir)

# If run as script (e.g., via `accelerate launch train_lora.py`), optionally accept simple args
if __name__ == '__main__':
    # Default: run the grid once. If you want fewer runs, edit `grid_options` above.
    run_experiments(grid_options)

In [ ]:
# Kaggle: run everything non-interactively (install, convert, configure accelerate, launch)
import subprocess
cmds = [
    'pip install -q transformers datasets accelerate peft sentence-transformers evaluate scikit-learn',
    'jupyter nbconvert --to script finetune_lora.ipynb --output train_lora.py',
    'accelerate config default',
    'accelerate launch train_lora.py -- --overwrite False'
]
# Execute commands sequentially; let failures raise so Kaggle job stops on error
for c in cmds:
    subprocess.check_call(c, shell=True)